# h-Louvain: NMI / AMI against ground truth

Set the two file paths in the next cell, then **Run All**.

**Input 1 — hypergraph file** (`HYPERGRAPH_FILE`): one hyperedge per row, node labels comma-separated (same format as `results_300_he.txt`).

**Input 2 — true clusters file** (`TRUE_CLUSTERS_FILE`): one cluster label per row, row `i` = the `i`-th node in ascending node order.

Settings: `hmod_tau = 1` (linear), `resolution = 1`, basic `h_louvain_community` execution (no last-step optimization).

In [3]:
from pathlib import Path

# --- inputs -----------------------------------------------------------------
ROOT = Path('C:/Users/ckeat/github-projects/Summer_Research/python_19_05_files/clustering_ML')

HYPERGRAPH_FILE = ROOT / 'data/synthetic_data/hypernetwork_form/edges/cluster_id1.txt'
TRUE_CLUSTERS_FILE  = ROOT / 'data/synthetic_data/true_clusters/cluster_id1.txt'

RANDOM_SEED = 123

In [4]:
import csv

import hypernetx as hnx
import hypernetx.algorithms.hypergraph_modularity as hmod
import h_louvain as hl
from sklearn.metrics import normalized_mutual_info_score as NMI
from sklearn.metrics import adjusted_rand_score as ARI

In [5]:
def load_hypergraph_from_file(filename):
    """One hyperedge per row, comma-separated node labels."""
    with open(filename, "r") as f:
        rd = csv.reader(f)
        lines = list(rd)
    Edges = []
    for line in lines:
        edge = [v.strip() for v in line if v.strip() != ""]
        if edge:
            Edges.append(edge)
    return hnx.Hypergraph(dict(enumerate(Edges)))


def load_true_clusters(filename):
    """One cluster label per row, in ascending node order."""
    with open(filename, "r") as f:
        return [line.strip() for line in f if line.strip() != ""]


def nodes_in_ascending_order(HG):
    """Node labels sorted numerically when they are all numbers, else lexicographically."""
    nodes = [str(v) for v in HG.nodes]
    try:
        return sorted(nodes, key=lambda x: int(x))
    except ValueError:
        return sorted(nodes)

In [6]:
HG = load_hypergraph_from_file(HYPERGRAPH_FILE)
gt = load_true_clusters(TRUE_CLUSTERS_FILE)
node_order = nodes_in_ascending_order(HG)

print("nodes in hypergraph :", len(node_order))
print("hyperedges          :", len(HG.edges))
print("ground-truth rows   :", len(gt))
print("ground-truth classes:", len(set(gt)))

if len(gt) != len(node_order):
    raise ValueError(
        f"{len(gt)} ground-truth rows but {len(node_order)} nodes in the hypergraph. "
        "Every node must have exactly one row (nodes appearing in no hyperedge are not in the hypergraph)."
    )

nodes in hypergraph : 1000
hyperedges          : 2213
ground-truth rows   : 1000
ground-truth classes: 3


In [7]:
%%time
# hmod_tau: w(d,c) = (c/d)^tau for c > d/2 else 0
# hmod_tau = 1 (linear), 0 (majority), "infinity" (strict)

hL = hl.hLouvain(HG, hmod_tau=1, resolution=1, random_seed=RANDOM_SEED)

CPU times: total: 1.28 s
Wall time: 1.46 s


In [8]:
%%time
# basic hLouvain algorithm execution (without the last_step optimization)

c = 0.3
b = 0.8
alphas = [1 - ((1 - b) ** i) for i in range(30)]

A, q2, alphas_out = hL.h_louvain_community(alphas=alphas, change_frequency=c)

print("final_alpha =", alphas_out[-1])
print("qH =", q2)
print("communities found =", len(A))

final_alpha = 1
qH = 0.501774714675782
communities found = 14
CPU times: total: 4.33 s
Wall time: 4.57 s


In [ ]:
# map the partition back to one label per node, in the same order as the ground-truth file
d = hmod.part2dict(A)
pred = [d[n] for n in node_order]

print("NMI =", NMI(gt, pred))
print("AdjRand =", ARI(gt, pred))
print("comm    =", len(A))
print("comm-gt =", len(set(gt)))

NMI = 0.3403890336332751
ARI = 0.20171895903233317
comm    = 14
comm-gt = 3
